# МЕТОДИЧЕСКИЙ ОТЧЕТ ПО ЛАБОРАТОРНОЙ РАБОТЕ
## «DCGAN: реализация, обучение и анализ архитектурных решений»

**Дисциплина:** Generative Adversarial Networks (ANL7311)  
**Тема:** Лекция 4 — DCGAN: глубокие свёрточные архитектуры и базовые техники стабилизации  
**Студент (ФИО):** Altynbek2004  
**Дата выполнения:** 2026-09-30  
**Набор данных:** MNIST ($28 \times 28$, grayscale, 60 000 изображений)

---

## 1. Цель работы
Реализовать архитектуру DCGAN «с нуля» на основе пяти архитектурных правил Radford et al. (2015), обучить её на наборе изображений MNIST, а также экспериментально проверить, как именно каждый архитектурный приём влияет на стабильность обучения и качество генерации.

## 2. Инициализация и архитектуры сетей по правилам Radford et al.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

# Воспроизводимость
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f'Используемое устройство: {device}')

# Правило инициализации весов Radford et al. (2015)
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

# Генератор (Generator)
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(z_dim, 256 * 7 * 7, bias=False),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.ReLU(True)
        )
        self.conv_blocks = nn.Sequential(
            # Блок 1: (256, 7, 7) -> (128, 14, 14)
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # Блок 2: (128, 14, 14) -> (64, 28, 28)
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # Финальный слой -> Tanh (значения в [-1, 1])
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1, bias=True),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 256, 7, 7)
        return self.conv_blocks(x)

# Дискриминатор (Discriminator)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            # Блок 1: без BatchNorm по правилам DCGAN
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Блок 2: с BatchNorm
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feats = self.conv_blocks(x)
        feats = feats.view(feats.size(0), -1)
        return self.classifier(feats)

## 3. Задание 1: Реализация и базовое обучение DCGAN (15 эпох)
Отобразим график сходимости потерь генератора и дискриминатора, а также визуализируем эволюцию сгенерированных изображений на фиксированном наборе векторов шума $z$ (25 примеров на эпохах 1, 8 и 15).

In [ ]:
from IPython.display import Image, display
print('1. График функций потерь G и D (Задание 1):')
display(Image(filename='images/lab3_task1_loss_curve.png'))

print('2. Сводная эволюция качества генерации (Эпоха 1 -> Эпоха 8 -> Эпоха 15):')
display(Image(filename='images/lab3_task1_evolution_grid.png'))

> **Ожидаемый результат Задания 1:**  
> Построен совместный график потерь $G$ и $D$. Получены и зафиксированы сетки изображений на первой, средней и последней эпохах. На 15 эпохе сформирована итоговая сетка из 25 четко узнаваемых, разнообразных рукописных цифр MNIST без фонового шума.

## 4. Задание 2: Абляционный эксперимент — удаление Batch Normalization
Сравнение функций потерь и итогового качества сгенерированных выборок при одинаковых гиперпараметрах для Baseline модели и модели без BatchNorm.

In [ ]:
print('1. Сравнение кривых потерь D и G (с BN vs без BN):')
display(Image(filename='images/lab3_task2_ablation_loss_comparison.png'))

print('2. Сравнение качества генерации на эпохе 15:')
display(Image(filename='images/lab3_task2_ablation_samples_comparison.png'))

> **Ожидаемый результат Задания 2:**  
> 1. Два сравнительных графика потерь на одних осях построены.  
> 2. Сетка сгенерированных изображений на одинаковом этапе сформирована.  
> 3. **Письменный вывод:** Удаление Batch Normalization приводит к резкому дисбалансу состязательного процесса. Из-за отсутствия нормализации промежуточных активаций дискриминатор деградирует ($D\_loss > 1.22$), а генератор лишается внятных обучающих сигналов ($G\_loss < 0.97$). Изображения без BN смазаны, содержат структурные дефекты и демонстрируют признаки частичного коллапса мод.

## 5. Задание 3: Исследование латентного пространства (Интерполяция)
Линейная интерполяция $z(t) = (1 - t) z_1 + t z_2$ для 3 независимых случайных пар векторов по 10 шагов.

In [ ]:
print('Линейная интерполяция в латентном пространстве (3 пары векторов):')
display(Image(filename='images/lab3_task3_latent_interpolation.png'))

> **Ожидаемый результат Задания 3:**  
> Построены 3 ряда изображений по 10 шагов.  
> **Вывод о гладкости и сравнение с VAE:**  
> 1. **Гладкость перехода:** Переход между символами происходит непрерывно. Промежуточные шаги представляют собой правдоподобный морфинг штрихов и контуров без пиксельного шума.  
> 2. **Сравнение с VAE:** Латентное пространство VAE теоретически более плотное и гарантированно не содержит разрывов за счет штрафа KL-дивергенции ($D_{KL}(q(z|x) \parallel p(z))$). Однако промежуточные кадры в VAE часто размыты из-за усредняющего MSE-лосса. В DCGAN генератор сохраняет максимальную контрастность и четкость на всех шагах интерполяции.

## 6. Итоговые выводы (5–7 предложений)
1. В ходе работы реализована и исследована сверточная архитектура DCGAN в соответствии с пятью правилами Radford et al. (2015).
2. Базовая модель успешно обучилась за 15 эпох на MNIST, обеспечив высокое качество и разнообразие генерируемых рукописных цифр.
3. Наиболее критичным архитектурным приемом для стабильности обучения оказался **Batch Normalization (пакетная нормализация)**.
4. Его удаление спровоцировало внутренний ковариационный сдвиг, деградацию дискриминатора ($D\_loss > 1.22$) и появление смазанных артефактов у генератора.
5. Исследование латентного пространства показало высокую степень семантической гладкости и непрерывности выученного генератором многообразия.
6. Все поставленные задачи и требования методических указаний выполнены в полном объеме.

## 7. Ответы на контрольные вопросы для устной защиты

**1. Почему в генераторе используется Tanh, а в дискриминаторе — Sigmoid, а не наоборот?**  
Генератор формирует изображение с нормализацией пикселей в $[-1, 1]$, поэтому симметричная функция $\tanh(x) \in (-1, 1)$ обеспечивает центрированные градиенты. Дискриминатор оценивает вероятность истинности выборки $D(x) \in (0, 1)$, что строго требует $\text{Sigmoid}$ для корректного расчета Binary Cross-Entropy Loss. Если поменять их местами, $\log D(x)$ сломается при отрицательных выходах Tanh, а Sigmoid в генераторе нарушит симметрию распределения пикселей.

**2. Что произойдёт с обучением, если полностью убрать Batch Normalization из обеих сетей одновременно?**  
Возникает неконтролируемый ковариационный сдвиг и экспоненциальный дрейф активаций слоев. Дискриминатор теряет чувствительность ($D\_loss$ растет выше 1.22), состязательный баланс разрушается, а генератор выдает размытые формы с дефектами и впадает в частичный коллапс мод.

**3. Почему в дискриминаторе используется LeakyReLU, а не обычный ReLU, и как это связано с градиентами?**  
Обычный ReLU при $x \le 0$ дает нулевую производную, что приводит к отмиранию нейронов («Dying ReLU»). Поскольку обучающий градиент генератора проходит сквозь дискриминатор (цепное правило $\nabla_{\theta_G} = \frac{\partial L}{\partial D} \cdot \frac{\partial D}{\partial G} \cdot \frac{\partial G}{\partial \theta_G}$), нулевой градиент от ReLU дискриминатора полностью остановил бы обучение генератора. $\text{LeakyReLU}(\alpha=0.2)$ обеспечивает непрерывный пропуск слабого градиентного сигнала.

**4. Что означает резкое падение функции потерь дискриминатора почти до нуля на графике обучения, и почему это плохой знак для генератора?**  
Это свидетельствует о том, что дискриминатор стал идеальным классификатором ($D(x_{real}) \approx 1, D(G(z)) \approx 0$). В этой зоне градиент сигмоиды насыщается и исчезает ($\nabla_x D(x) \to \mathbf{0}$), из-за чего генератор перестает получать обратную связь и его обучение «замерзает».

**5. Как латентное пространство, изученное GAN, соотносится с латентным пространством VAE — какое из них теоретически должно быть более гладким и почему?**  
Теоретически более гладким является пространство VAE, поскольку оно принудительно регуляризуется штрафом $D_{KL}(q(z|x) \parallel p(z))$ к стандартному нормальному распределению, исключая незаполненные пустоты. В GAN пространство оптимизируется косвенно; однако GAN обеспечивает существенно более высокую контрастность и детализацию без размытия, свойственного VAE.

**6. Почему параметры оптимизатора Adam (в частности, $\beta_1=0.5$ вместо стандартного 0.9) особенно важны именно для GAN?**  
В динамической минимаксной игре GAN поверхность функции потерь непрерывно сдвигается каждым игроком. Высокий моментум $\beta_1 = 0.9$ использует устаревшую инерцию прошлых шагов, вызывая перелеты (overshooting) и паразитные циклические осцилляции. Значение $\beta_1 = 0.5$ уменьшает память оптимизатора и стабилизирует сходимость.

**7. Если генератор во время обучения начал выдавать почти одинаковые изображения независимо от $z$ — как это явление называется и что из разобранного на Лекции 4 могло бы отчасти помочь?**  
Это явление называется **коллапсом мод (Mode Collapse)**. Помогает: 1) Batch Normalization (стохастический шум мини-батча не дает представлениям схлопнуться в точку); 2) корректная инициализация весов (нормальное распределение со std=0.02); 3) сбалансированная динамика оптимизатора с $\beta_1=0.5$.